# Target definition and outcome quality

This notebook considers confirmed deliveries, other unconfirmed delivery records, and cancellation separately. An absent recorded delivery date does not prove a shipment was never delivered.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

orders = pd.read_csv(Path("data/raw") / "olist_orders_dataset.csv", parse_dates=[
    "order_purchase_timestamp", "order_delivered_customer_date",
    "order_estimated_delivery_date"
])
assert orders.order_id.is_unique
cohort = orders.loc[
    orders.order_purchase_timestamp.between("2017-01-01", "2018-08-31 23:59:59")
    & orders.order_estimated_delivery_date.lt(pd.Timestamp("2018-09-01"))
].copy()
actual = cohort.order_delivered_customer_date.dt.normalize()
expected = cohort.order_estimated_delivery_date.dt.normalize()
on_time = actual.notna() & actual.le(expected)
late = actual.notna() & actual.gt(expected)
canceled_no_delivery = actual.isna() & cohort.order_status.eq("canceled")
cohort["delivery_outcome"] = np.select(
    [on_time, late, canceled_no_delivery, actual.isna()],
    ["Delivered on time", "Delivered late", "Canceled without delivery date", "Other without delivery date"],
    default="Unknown"
)
display(cohort.delivery_outcome.value_counts(dropna=False))
display(pd.crosstab(cohort.order_status, cohort.delivery_outcome, margins=True))

## Confirmed-delivery target

We restrict the binary classification experiment to records with both confirmed delivered status and a valid actual delivery timestamp. This is an explicit selection restriction, not an imputation that all unresolved orders were on-time.

In [ ]:
confirmed = cohort.loc[
    cohort.order_status.eq("delivered")
    & cohort.order_delivered_customer_date.notna()
    & cohort.order_estimated_delivery_date.notna()
].copy()
confirmed["late_delivery"] = (
    confirmed.order_delivered_customer_date.dt.normalize()
    > confirmed.order_estimated_delivery_date.dt.normalize()
).astype(int)
assert confirmed.order_id.is_unique
print("Cohort:", len(cohort))
print("Confirmed:", len(confirmed))
print("Excluded:", len(cohort) - len(confirmed))
print("Late confirmed deliveries:", confirmed.late_delivery.sum())
print("Observed positive rate:", round(confirmed.late_delivery.mean(), 4))

## Limitations

The classifier is conditioned on a confirmed recorded delivery. Right censoring, inconsistent final statuses, and canceled or unconfirmed orders can limit transfer to a live fulfilment system. A genuine failure-to-meet-commitment model would require new target definitions and reliable observation snapshots.